In [ ]:
!pip install gensim


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.7/26.7 MB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 8.7 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: scipy
    Found existing installation: scipy 1.14.1
    Uninstalling scipy-1.14.1:
      Successfully uninstalled scipy-1.14.1


In [ ]:
!pip install numpy

In [ ]:
pip install numpy gensim scipy


In [ ]:
!pip install -U spacy

In [ ]:
!python -m spacy download es_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 32.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import spacy

!python -m spacy download es_core_news_sm  # Download the Spanish model

nlp = spacy.load("es_core_news_sm")  # Load the model

ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


In [ ]:
!pip install -U spacy

In [ ]:
# Librerías esenciales
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# Utilidades de PyTorch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

# Procesamiento de texto y NLP
import re
import unicodedata
import spacy
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import Word2Vec

# Manejo de datos y estadísticas
import numpy as np
from sklearn.model_selection import train_test_split
from collections import Counter

# Visualización
import matplotlib.pyplot as plt


In [ ]:
# Contadores de categorías
num_negativos = 0
num_positivos = 0
num_neutro = 0

datos = []
etiquetas = []

with open('tweets2.txt', 'r', encoding='utf-8') as file:
    lines = file.readlines()

for line in lines:
    parts = line.strip().split('\t', 1)  # Separamos por el primer tabulador
    if len(parts) < 2:
        continue  # Ignorar líneas mal formateadas

    etiqueta = parts[0].lower()  # Convertimos a minúsculas para evitar inconsistencias
    texto = parts[1]  # El tweet

    if etiqueta == 'n':
        etiquetas.append(0)
        num_negativos += 1
    elif etiqueta == 'p':
        etiquetas.append(1)
        num_positivos += 1
    elif etiqueta == 'neu':
        etiquetas.append(2)
        num_neutro += 1
    else:
        continue  # Ignorar etiquetas desconocidas

    datos.append(texto)

# Mostrar conteo de categorías
print(f"Total de tweets negativos (N): {num_negativos}")
print(f"Total de tweets positivos (P): {num_positivos}")
print(f"Total de tweets neutros (NEU): {num_neutro}")

# Verificar que los datos se han cargado correctamente
print("[Ejemplo de datos]")
print("\n".join(datos[:10]))  # Mostrar los primeros 5 textos
print(etiquetas[:10])  # Mostrar las primeras 5 etiquetas

# Convertir las etiquetas a tensor de tipo float32 para la pérdida categórica
etiquetas = torch.tensor(etiquetas, dtype=torch.float32)


Total de tweets negativos (N): 730
Total de tweets positivos (P): 457
Total de tweets neutros (NEU): 264
[Ejemplo de datos]
No mames este pinche dolor que pedo? ya mejor llévame Diosito.
@leomall2018 Según yo era como aviso, pero ahora sí ya es oficial
@benshorts a juzgar por mis comportamientos autodestructivos en las relaciones, aún quiero serlo
#BuenosDias mundo Twittero ya desperté y estoy listo para vivir un dia mas #ExcelenteMartes
No pude resolver el rompecabezas en Los rios de Alicia y ahora muero de tristeza
o sea ... me urge un Dr. @Rocktor101 (escuchó un programa de males digestivos y así)
@natyamezcua hay pues hay que ver que hacemos para vernos, te extraño, no quiero que te alejes
@Eudemonologia87 yo igual y ni así. Este año seré tipo marasalvatrucha para ver si así me traen algo.
@shakira no te puedo llevar en mi bicicleta. Me caí y está descompuesta. Happy to walk with you, tho.
@MartinCaligaris Tu forma tan linda de decir: Borris Chavez!!! Gracias por el saludo, ya pron

In [ ]:
# Descargar modelo de spaCy y stopwords de NLTK
nlp = spacy.load("es_core_news_sm")
nltk.download('stopwords')
stop_words = set(stopwords.words('spanish'))
stop_words.update({"q", "xd", "aja", "jaja", "jajaja", "ja", "xq", "k","oh"})

# Función para limpiar texto
def limpiar_texto(texto):
    # Normaliza el texto eliminando caracteres especiales y acentos
    texto = unicodedata.normalize('NFD', texto).encode('ascii', 'ignore').decode('utf-8')

    # Convierte el texto a minúsculas
    texto = texto.lower()

    # Elimina menciones (@usuario) y paginas web
    texto = re.sub(r"@\w+|https?://\S+", "", texto)

    # Elimina el '#' de los hashtags
    texto = re.sub(r"#", "", texto)

    # Reduce repeticiones excesivas de caracteres
    texto = re.sub(r'(.)\1{2,}', r'\1', texto)

    # Elimina caracteres que no sean letras, números o espacios
    texto = re.sub(r"[^a-zA-Z0-9\s]", "", texto)

    # Filtra palabras vacías (stop words), asumiendo que 'stop_words' está definido previamente
    palabras_limpias = [palabra for palabra in texto.split() if palabra not in stop_words]

    return palabras_limpias

# Leer datos desde el archivo
datos = []
etiquetas = []

with open('tweets2.txt', 'r', encoding='utf-8') as file:
    lines = file.readlines()

for line in lines:
    parts = line.strip().split('\t', 1)
    if len(parts) < 2:
        continue
    etiqueta = parts[0].lower()
    texto = parts[1]
    if etiqueta == 'n':
        etiquetas.append(0)
    elif etiqueta == 'p':
        etiquetas.append(1)
    elif etiqueta == 'neu':
        etiquetas.append(2)
    else:
        continue
    datos.append(texto)

datos_limpiados = [limpiar_texto(texto) for texto in datos]

# Entrenar Word2Vec
embedding_dim = 200
modelo_word2vec = Word2Vec(sentences=datos_limpiados, vector_size=embedding_dim, window=6, min_count=2, workers=8)

# Construcción del vocabulario
num_palabras = Counter(word for texto in datos_limpiados for word in texto)
vocabulario = {palabra: i+1 for i, (palabra, frecuencia) in enumerate(num_palabras.items())}

# Crear matriz de embeddings, Llena la matriz con los embeddings del modelo Word2Vec
embedding_matrix = torch.zeros(len(vocabulario) + 1, embedding_dim)
for palabra, i in vocabulario.items():
    if palabra in modelo_word2vec.wv:
        embedding_matrix[i] = torch.tensor(modelo_word2vec.wv[palabra])

# Convertir texto en secuencias de índices
def texto_a_indices(texto):
    return [vocabulario.get(palabra, 0) for palabra in texto]

datos_indices = [torch.tensor(texto_a_indices(texto)) for texto in datos_limpiados]
datos_embedded = pad_sequence(datos_indices, batch_first=True, padding_value=0)

# Imprimir ejemplo de datos procesados
print("Ejemplo de datos procesados:")
for i, (texto_original, texto_procesado, indices) in enumerate(zip(datos[:5], datos_limpiados[:5], datos_indices[:5])):
    print(f"Tweet {i+1}:")
    print(f"  Original: {texto_original}")
    print(f"  Procesado: {texto_procesado}")
    print(f"  Índices: {indices.tolist()}")
    print()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Sebastian\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Ejemplo de datos procesados:
Tweet 1:
  Original: No mames este pinche dolor que pedo? ya mejor llévame Diosito.
  Procesado: ['mames', 'pinche', 'dolor', 'pedo', 'mejor', 'llevame', 'diosito']
  Índices: [1, 2, 3, 4, 5, 6, 7]

Tweet 2:
  Original: @leomall2018 Según yo era como aviso, pero ahora sí ya es oficial
  Procesado: ['segun', 'aviso', 'ahora', 'si', 'oficial']
  Índices: [8, 9, 10, 11, 12]

Tweet 3:
  Original: @benshorts a juzgar por mis comportamientos autodestructivos en las relaciones, aún quiero serlo
  Procesado: ['juzgar', 'comportamientos', 'autodestructivos', 'relaciones', 'aun', 'quiero', 'serlo']
  Índices: [13, 14, 15, 16, 17, 18, 19]

Tweet 4:
  Original: #BuenosDias mundo Twittero ya desperté y estoy listo para vivir un dia mas #ExcelenteMartes
  Procesado: ['buenosdias', 'mundo', 'twittero', 'desperte', 'listo', 'vivir', 'dia', 'mas', 'excelentemartes']
  Índices: [20, 21, 22, 23, 24, 25, 26, 27, 28]

Tweet 5:
  Original: No pude resolver el rompecabezas en Los

In [ ]:
# Convertir a tensores
X = datos_embedded
y = torch.tensor(etiquetas, dtype=torch.long)

# División de datos
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

class Dato(Dataset):
    def __init__(self, X, y):
        self.X = X  # Datos de entrada
        self.y = y  # Etiquetas

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Creación de conjuntos de entrenamiento y prueba usando la clase Dato
train_dataset = Dato(X_train, y_train)
test_dataset = Dato(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [ ]:
# Definir modelo BiLSTM
class BiLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, embedding_matrix):
        super(BiLSTM, self).__init__()
        #hidden_dim Número de neuronas en la capa oculta de la LSTM.
        #output_dim Número de clases en la salida
        #embedding_matrix Matriz  de embeddings Word2Vec

        self.embedding = nn.Embedding.from_pretrained(embedding_matrix, freeze=False)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=1, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(hidden_dim * 2, output_dim) #para que sea bidiomencional

        self.dropout = nn.Dropout(0.5)#para el sobre ajuste

    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, _ = self.lstm(embedded)
        final_output = self.fc(lstm_out[:, -1, :]) #Seleccionamos la ultima palabra de cada secuencia
        return final_output

        #salida de la LSTM ya tiene la información acumulada de todas las palabras anteriores.

# Inicializar modelo
vocab_size = len(vocabulario) + 1
hidden_dim = 128
output_dim = 3  # Tres clases

model = BiLSTM(vocab_size, embedding_dim, hidden_dim, output_dim, embedding_matrix)

In [ ]:
# Configurar entrenamiento
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

def train_model(model, train_loader, criterion, optimizer, epochs=65):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch in train_loader:
            X_batch, y_batch = batch # Obtener datos de entrada (X) y etiquetas (y)
            optimizer.zero_grad() # Reiniciar
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)# Calcular la perdida entre predicción y etiqueta real
            loss.backward()
            optimizer.step() # Actualizar los pesos del modelo
            total_loss += loss.item()
        print(f"Epoch {epoch+1}, Loss: {total_loss}")

train_model(model, train_loader, criterion, optimizer)


Epoch 1, Loss: 39.95357871055603
Epoch 2, Loss: 38.76382803916931
Epoch 3, Loss: 37.865947902202606
Epoch 4, Loss: 37.76427739858627
Epoch 5, Loss: 37.23024547100067
Epoch 6, Loss: 30.872415363788605
Epoch 7, Loss: 19.98055812716484
Epoch 8, Loss: 16.70804050564766
Epoch 9, Loss: 15.089413493871689
Epoch 10, Loss: 14.094696059823036
Epoch 11, Loss: 13.682247757911682
Epoch 12, Loss: 13.306145586073399
Epoch 13, Loss: 13.270704299211502
Epoch 14, Loss: 12.721454560756683
Epoch 15, Loss: 12.669612258672714
Epoch 16, Loss: 11.705655559897423
Epoch 17, Loss: 9.818862468004227
Epoch 18, Loss: 7.815605241805315
Epoch 19, Loss: 6.45492609590292
Epoch 20, Loss: 5.002078324556351
Epoch 21, Loss: 3.551277754828334
Epoch 22, Loss: 2.588130487129092
Epoch 23, Loss: 2.1526241321116686
Epoch 24, Loss: 1.6117128059267998
Epoch 25, Loss: 1.4773980416357517
Epoch 26, Loss: 1.3697910020127892
Epoch 27, Loss: 1.3186820978298783
Epoch 28, Loss: 1.21134820766747
Epoch 29, Loss: 1.0549501162022352
Epoch 30,

In [ ]:
# Evaluar el modelo
correctos = 0
total = 0
with torch.no_grad():
    model.eval()
    for inputs, labels in test_loader:
        outputs = model(inputs)
        predicted = torch.argmax(outputs, dim=1)  # Obtener la clase con mayor probabilidad
        correctos += (predicted == labels).sum().item()
        total += labels.size(0)

accuracy = correctos / total
print(f"Accuracy del modelo: {accuracy * 100:.2f}%")

Accuracy del modelo: 51.55%


In [ ]:
def clasificacion_datos(text):
    encoded = texto_a_indices(limpiar_texto(text))
    encoded_tensor = torch.tensor(encoded, dtype=torch.long)  # Eliminamos unsqueeze(0)

      # Aplicar padding para asegurar que la secuencia tenga el mismo tamaño esperado por el modelo
    padded = pad_sequence([encoded_tensor], batch_first=True, padding_value=0)
     # Mover el tensor a CPU para asegurar compatibilidad
    tensor_input = padded.to(torch.device("cpu"))

    print("Forma de tensor_input:", tensor_input.shape)  # Debug para verificar dimensiones

    model.eval()
    with torch.no_grad():
        output = model(tensor_input).squeeze()
        predicted_class = torch.argmax(output).item()

    clases = ["Negativo", "Neutral", "Positivo"]
    return clases[predicted_class]

# Pruebas de clasificación
ejemplos = [
    # Negativos (N)
    "No mames este pinche dolor que pedo? ya mejor llévame Diosito.",
    "Puta madre lo que me faltaba enfermarme #QuieroLlorar",
    "Estoy tan cansado de todo, nada me sale bien últimamente.",
    "No soporto más este estrés, todo es un caos en mi vida.",
    "Me despidieron del trabajo, no sé qué voy a hacer ahora.",
    "No tengo ganas de nada, este día ha sido una porquería.",

    # Positivos (P)
    "Pagaré una maleta extra para todos nuestros nuevos momentos juntos",
    "#BuenosDias mundo Twittero ya desperté y estoy listo para vivir un dia mas #ExcelenteMartes",
    "Hoy fue un día increíble, logré todo lo que me propuse.",
    "Qué felicidad, por fin llegó el fin de semana, a disfrutar!",
    "Amo estos momentos en familia, son lo mejor que tengo.",
    "Hoy me siento en paz y lleno de gratitud, todo fluye bien.",
    # Neutros (NEU)
    "@leomall2018 Según yo era como aviso, pero ahora sí ya es oficial.",
    "4:18 AM. Que gran dilema. ¿Despertar a mamá para que me baje a abrirme o dejar que siga durmiendo? Pd. Venga saliendo del trabajo.",
    "@natyamezcua hay pues hay que ver que hacemos para vernos, te extraño, no quiero que te alejes",
    "Si sabían que Broca y Wernicke son mejores amigos, cuando Wernicke se confunde, Broca se traba,",
    "Si quieren que aprenda algo de eso dejen esa materia sola un semestre completo, si no como."
]
for ejemplo in ejemplos:
    print(f"Texto: {ejemplo}\nPredicción: {clasificacion_datos(ejemplo)}\n")

Forma de tensor_input: torch.Size([1, 7])
Texto: No mames este pinche dolor que pedo? ya mejor llévame Diosito.
Predicción: Negativo

Forma de tensor_input: torch.Size([1, 5])
Texto: Puta madre lo que me faltaba enfermarme #QuieroLlorar
Predicción: Negativo

Forma de tensor_input: torch.Size([1, 5])
Texto: Estoy tan cansado de todo, nada me sale bien últimamente.
Predicción: Positivo

Forma de tensor_input: torch.Size([1, 5])
Texto: No soporto más este estrés, todo es un caos en mi vida.
Predicción: Negativo

Forma de tensor_input: torch.Size([1, 5])
Texto: Me despidieron del trabajo, no sé qué voy a hacer ahora.
Predicción: Positivo

Forma de tensor_input: torch.Size([1, 4])
Texto: No tengo ganas de nada, este día ha sido una porquería.
Predicción: Positivo

Forma de tensor_input: torch.Size([1, 6])
Texto: Pagaré una maleta extra para todos nuestros nuevos momentos juntos
Predicción: Neutral

Forma de tensor_input: torch.Size([1, 9])
Texto: #BuenosDias mundo Twittero ya desperté y est